# RAMS AI Camera, Film 2: "See. Detect. Protect." (render on Kaggle)

Renders the applications film on Kaggle's free **2x T4** GPUs, runs the AI-vision post (perception scan, HUD,
counters, titles, logo), adds the music and cuts one clip per shot.
Full steps: `docs/apps_film/RENDER_ON_KAGGLE.md` in the project.

1. **Settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**.
2. **Music:** upload `In_Motion.mp3` once as a private Kaggle dataset and **Add Input** it. It's found
   automatically; without it the film is made silent.
3. Set `MODE` below: `"preview"` (960x540, about 3-5 h) or `"final"` (1920x1080; may need 2 sessions).
4. **Save Version > Save & Run All (Commit).** It runs in the background (up to 12 h).

**Resuming:** if a run stops early, open the notebook, **Add Input > Your Work >** this notebook's latest version,
and **Save & Run All** again. Finished frames are reused.

**Results:** the version's **Output** tab, `film2/` (MP4 + clips zip).

In [ ]:
# ---- settings -------------------------------------------------------------------------------
MODE = "preview"       # "preview" (960x540, 24 samples) or "final" (1920x1080, 64 samples)
REPO = "https://github.com/Yashism/Blender-CC.git"
BRANCH = "claude/rams-ai-camera-launch-film"
MAX_HOURS = 10.8       # Kaggle stops a session at 12 h: stop rendering in time to run the post and save
BLENDER_URL = "https://download.blender.org/release/Blender4.5/blender-4.5.14-linux-x64.tar.xz"
RES, SAMPLES, POST_SIZE = {"preview": ("960x540", 24, "960x540"), "final": ("1920x1080", 64, "1920x1080")}[MODE]

In [ ]:
# ---- setup: GPUs, Blender 4.5, project, earlier frames -----------------------------------------
import glob, os, shutil, subprocess, time
WORK = "/kaggle/working"
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
PROJ = f"{TMP}/blender-cc"
FRAMES = f"{WORK}/frames_{MODE}"
OUT = f"{WORK}/film2"
LOGS = f"{WORK}/logs"
for d in (FRAMES, OUT, LOGS):
    os.makedirs(d, exist_ok=True)
gpus = [l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.strip()]
print("\n".join(gpus) or "NO GPU: set Accelerator to GPU T4 x2")
assert gpus, "no GPU"
BL = f"{TMP}/blender/blender"
if not os.path.exists(BL):
    subprocess.run(f"wget -q -O {TMP}/bl.tar.xz {BLENDER_URL} && mkdir -p {TMP}/blender && "
                   f"tar -xf {TMP}/bl.tar.xz -C {TMP}/blender --strip-components=1 && rm {TMP}/bl.tar.xz", shell=True, check=True)
subprocess.run("apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 ffmpeg > /dev/null 2>&1",
               shell=True)
print(subprocess.run([BL, "--version"], capture_output=True, text=True).stdout.splitlines()[0])
token = ""
try:                    # only if the repo is private: Add-ons > Secrets > GITHUB_TOKEN
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
if not os.path.isdir(PROJ):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, PROJ], check=True)
print(subprocess.run(["git", "-C", PROJ, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
# resume: copy frames / passes / anchors from an earlier version attached as input
n = 0
for p in glob.glob(f"/kaggle/input/**/frames_{MODE}/**/*", recursive=True):
    if os.path.isfile(p):
        rel = p.split(f"frames_{MODE}/", 1)[1]
        dst = os.path.join(FRAMES, rel)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(p, dst)
            n += 1
print("files reused:", n, "| frames present:", len(glob.glob(FRAMES + "/f_*.jpg")))
# music
mp3 = sorted(glob.glob("/kaggle/input/**/*In_Motion*.mp3", recursive=True) + glob.glob("/kaggle/input/**/*in_motion*.mp3", recursive=True))
MUSIC = ""
if mp3:
    os.makedirs(f"{PROJ}/assets/music", exist_ok=True)
    shutil.copy(mp3[0], f"{PROJ}/assets/music/In_Motion.mp3")
    subprocess.run(["python3", f"{PROJ}/scripts/apps/music_edit.py"], check=True)
    MUSIC = f"{PROJ}/assets/music/In_Motion_film2.wav"
print("music:", MUSIC or "NOT FOUND (the film will be silent): add your In_Motion.mp3 dataset as input")

In [ ]:
# ---- render: one Blender per GPU, each takes every n-th frame -----------------------------------
import sys
sys.path.insert(0, f"{PROJ}/scripts")
from apps.timing import RENDER_END
SCRIPT = f"{PROJ}/scripts/apps/film2.py"
base = [BL, "-b", "--factory-startup", "--python-exit-code", "1", "-P", SCRIPT, "--", "--render", FRAMES,
        "--res", RES, "--samples", str(SAMPLES), "--device", "gpu", "--jpeg", "--pass-scale", "0.5"]
ng = len(gpus)
procs = []
for i in range(ng):
    lf = open(f"{LOGS}/gpu{i}.log", "a")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
    procs.append((subprocess.Popen(base + ["--max-hours", str(MAX_HOURS), "--slice", f"{i}/{ng}"], stdout=lf,
                                   stderr=subprocess.STDOUT, env=env), i))
    time.sleep(120)       # stagger the scene builds (the first also writes anchors.json)
t0 = time.time()
while any(p.poll() is None for p, _ in procs):
    time.sleep(300)
    done = len(glob.glob(FRAMES + "/f_*.jpg"))
    last = []
    for _, i in procs:
        ls = [l for l in open(f"{LOGS}/gpu{i}.log", errors="ignore") if l.startswith("[film2] frame")]
        last.append(f"gpu{i}: " + (ls[-1].strip() if ls else "starting"))
    print(f"{(time.time() - t0) / 3600:.2f} h | {done}/{RENDER_END} frames\n  " + "\n  ".join(last), flush=True)
print("exit codes:", [p.returncode for p, _ in procs])

In [ ]:
# ---- post: AI-vision scan, HUD, titles, logo, music -> MP4 + per-shot clips ----------------------
done = len(glob.glob(FRAMES + "/f_*.jpg"))
if done < RENDER_END:
    print(f"{done}/{RENDER_END} frames. Run again (Add Input > this notebook's output, then Save & Run All) to render the rest.")
else:
    film = f"{OUT}/RAMS_AI_Camera_film2_{MODE}.mp4"
    subprocess.run(["python3", f"{PROJ}/scripts/apps/sfx_film2.py"], check=True)      # sound design
    cmd = ["python3", f"{PROJ}/scripts/apps/post_film2.py", "--src", FRAMES, "--out", film, "--size", POST_SIZE, "--clips",
           "--jobs", str(os.cpu_count() or 2)]
    cmd += ["--music", MUSIC] if MUSIC else ["--no-music"]
    rc = subprocess.run(cmd, cwd=PROJ).returncode
    print("post exit code", rc)
    if MODE == "final":       # keep the output small: the frames are no longer needed once the film is made
        print("tip: delete frames_final from the output after downloading if you need the space")
    for p in sorted(glob.glob(OUT + "/**", recursive=True)):
        if os.path.isfile(p) and not p.endswith(".png"):
            print(f"  {os.path.getsize(p) / 1e6:8.1f} MB  {p}")